## External Validation of LASSO Model

**Dataset:** GSE42568: 104 breast cancer + 17 normal breast tissue transcriptomic profiles (Affymetrix U133 Plus 2.0).

**Objective** Utilize the saved models from analysis.ipynb and recreate a similar environment with an external dataset for validation of the model.

In [1]:
import os
import numpy as np
import pandas as pd
import joblib

#accuracy alone can be misleading, we'd want to add classification reports, conf_matrix
#as well as Receiving Operator Characteristic - Area Under the Curve (ROC_AUC)
#ROC_AUC is used to determine how well a binary classification model (which this is)
#distinguishes between two classes regardless of classification thresholds (which is 0.5 here)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)

EXTERNAL_DATA_PATH = "data/GSE42568_series_matrix.txt"
EXPORT_DIR         = "exported_model"


## Data Loading

A similar process to earlier

In [2]:
#load exported assets, scaler, model, feature list
scaler   = joblib.load(os.path.join(EXPORT_DIR, "training_scaler.joblib"))
model    = joblib.load(os.path.join(EXPORT_DIR, "lasso_1se_model.joblib"))
features = joblib.load(os.path.join(EXPORT_DIR, "retained_features.joblib"))

print("  LOADED TRAINING ARTIFACTS")
print(f"  Model type         : {type(model).__name__}")
print(f"  Model alpha        : {model.alpha:.6f}")
print(f"  # retained probes  : {len(features)}")
print(f"  Scaler n_features  : {scaler.n_features_in_}")
print(f"  Retained features  : {features}")

  LOADED TRAINING ARTIFACTS
  Model type         : Lasso
  Model alpha        : 0.129748
  # retained probes  : 15
  Scaler n_features  : 11142
  Retained features  : ['201650_at', '202014_at', '202988_s_at', '203549_s_at', '205478_at', '206243_at', '209550_at', '209686_at', '210964_s_at', '212112_s_at', '218124_at', '219398_at', '219738_s_at', '221009_s_at', '266_s_at']


In [4]:
#Read GEO file and extract labels, similar process to earlier, with a different parsing method for target 
#(since they use a different format)

y_test = None
sample_ids = None
matrix_start_line = None

with open(EXTERNAL_DATA_PATH, "r") as f:
    for line_num, line in enumerate(f, start=1):
        #this one uses tissure:normal, tissure:cancer
        if line.startswith("!Sample_characteristics_ch1") and "tissue:" in line:
            fields = line.strip().split("\t")[1:]
            y_test = np.array(
                [0 if "normal breast" in v else 1 for v in fields],
                dtype=int,
            )
        if line.startswith("!Sample_geo_accession"):
            sample_ids = [
                v.strip().strip('"') for v in line.strip().split("\t")[1:]
            ]
        if line.startswith("!series_matrix_table_begin"):
            matrix_start_line = line_num
            break

n_normal = int((y_test == 0).sum())
n_tumor  = int((y_test == 1).sum())
print(f"External dataset labels parsed:  {len(y_test)} samples  "
      f"({n_normal} normal, {n_tumor} tumor)")

External dataset labels parsed:  121 samples  (17 normal, 104 tumor)


In [ ]:
#load it as a DF
expr = pd.read_csv(
    EXTERNAL_DATA_PATH,
    sep="\t",
    skiprows=matrix_start_line,
    index_col=0,
)

if expr.index[-1].startswith("!"):
    expr = expr.iloc[:-1]

#transpose
X_test_df = expr.T
X_test_df = X_test_df.apply(pd.to_numeric, errors="coerce").fillna(0)

print(f"External expression matrix loaded: {X_test_df.shape}  (samples × probes)")

External expression matrix loaded: (121, 54675)  (samples × probes)


## Feature Alignment

The external dataset likely has different set of PROBE_IDs than the one we trained on.

We need to:
1. Keep ONLY the columns matching retained features
2. ZERO fill any retained features that did not exist in dataset (if performance is still good after this, it suggests model is doing very well)
3. Reorder columns to match training feature order

**Important**: we must provide the FULL scaler feature set, not just the
retained ones.  The training scaler was fit on ALL variance-filtered
features (~27 000+).  We first need to build a matrix with ALL those
columns (filling missing ones with 0) so that .transform() works.
Then we subset to the retained features AFTER scaling.


In [6]:
external_probes = set(X_test_df.columns)
retained_set    = set(features)

present  = retained_set & external_probes
missing  = retained_set - external_probes

print(f"Feature alignment:")
print(f"  Retained features     : {len(features)}")
print(f"  Found in external data: {len(present)}")
print(f"  Missing (zero-filled) : {len(missing)}")
if missing:
    print(f"    → {sorted(missing)}")

Feature alignment:
  Retained features     : 15
  Found in external data: 15
  Missing (zero-filled) : 0


In [ ]:
scaler_features = list(scaler.feature_names_in_) if hasattr(scaler, "feature_names_in_") else None

if scaler_features is not None:
    # build a DataFrame aligned to ALL scaler features
    X_test_full = pd.DataFrame(0.0, index=X_test_df.index, columns=scaler_features)

    # fill in values for probes that exist in both datasets.
    # we iterate column-by-column to avoid alignment issues between
    # the scaler-ordered and external-ordered DataFrames.
    common_full = list(set(scaler_features) & external_probes)
    for col in common_full:
        X_test_full[col] = X_test_df[col].values

    print(f"Full scaler alignment : {len(scaler_features)} columns "
          f"({len(common_full)} found, {len(scaler_features) - len(common_full)} zero-filled)")
    print()

    # we use .transform() — NOT .fit_transform() — so the centering and
    # scaling parameters come exclusively from the training data.
    X_test_scaled_full = scaler.transform(X_test_full)

print(f"Scaled test matrix shape: {X_test_scaled_full.shape}")
print()

  Full scaler alignment : 11142 columns (11136 found, 6 zero-filled)

Scaled test matrix shape: (121, 11142)



## Evaluate

A Lasso model outputs continuous predictions between [0,1]
Threshold is set at 0.5 as standard binary classification does
This can be adjusted at your own discretion though

This Dataset utilized the platform GLP570, according to my research online, this should be fine since 570 is simply a superset of GLP97.

In [8]:
y_scores = model.predict(X_test_scaled_full)
y_pred   = (y_scores > 0.5).astype(int)

acc     = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_scores)
cm      = confusion_matrix(y_test, y_pred)

# derive Sensitivity and Specificity from the confusion matrix
TN, FP, FN, TP = cm.ravel()
sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0   # recall for Tumor
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0   # recall for Normal

In [10]:

print("  EXTERNAL VALIDATION RESULTS  (GSE42568 — imbalanced)")
print(f"  Accuracy    : {acc:.4f}   (may be inflated by class imbalance)")
print(f"  ROC-AUC     : {roc_auc:.4f}")
print(f"  Sensitivity : {sensitivity:.4f}   (Tumor recall = TP / [TP + FN])")
print(f"  Specificity : {specificity:.4f}   (Normal recall = TN / [TN + FP])")
print("  Classification Report:")
print(
    classification_report(
        y_test, y_pred,
        target_names=["Normal", "Tumor"],
        zero_division=0,
    )
)

print("  Confusion Matrix:")
print(f"                  Predicted Normal   Predicted Tumor")
print(f"  Actual Normal         {TN:>5}             {FP:>5}")
print(f"  Actual Tumor          {FN:>5}             {TP:>5}")


  EXTERNAL VALIDATION RESULTS  (GSE42568 — imbalanced)
  Accuracy    : 0.9752   (may be inflated by class imbalance)
  ROC-AUC     : 0.9010
  Sensitivity : 1.0000   (Tumor recall = TP / [TP + FN])
  Specificity : 0.8235   (Normal recall = TN / [TN + FP])
  Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      0.82      0.90        17
       Tumor       0.97      1.00      0.99       104

    accuracy                           0.98       121
   macro avg       0.99      0.91      0.94       121
weighted avg       0.98      0.98      0.97       121

  Confusion Matrix:
                  Predicted Normal   Predicted Tumor
  Actual Normal            14                 3
  Actual Tumor              0               104


Struggles a little on normal samples, but since we have so little in this dataset, this may more be an issue on the test set rather than the model. Performance overall is still good. 14/17 on a massve minority set is still good i think. Note the **precision** score of 1 on normal samples, this means, even though it labels some normals as tumors (False Positives) it did not make an error on the much riskier false negatives.